# Stocks with ≥ 10,000% returns — drawdown and time

Standalone scan of the NSE parquet universe. It does **not** use the V5 strategy signals.

**Definition of 10,000%:** close reaches **101×** the entry close (`return = 100`, i.e. +10,000%).

**Path used:** buy at the **expanding minimum close** (lowest close so far). The run ends on the **first later close** that is ≥ 101× that low. Max drawdown is measured **only between entry and that first hit** — not after the target is already reached.

- `MaxDrawdownClose` — peak-to-trough using closes (`min(close / cummax(close) − 1)`)
- `MaxDrawdownLow` — trough using daily lows vs a running peak of `max(High, Close)` (worse of the two; what you can actually mark)

Days are reported as **trading sessions** (`HitIndex − EntryIndex`) and **calendar days**.

Corporate-action caveat: prices are used as stored. Unadjusted splits/bonus issues can create or destroy 101× moves. Rows with a very low entry price are flagged.


In [ ]:
from pathlib import Path
import warnings, gc, os, sys
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from quant.config import DATA_DIR as QUANT_DATA_DIR, PARQUET_DATA_DIR

DATA_DIR = PARQUET_DATA_DIR
OUT_DIR = QUANT_DATA_DIR / "results" / "hundred_baggers"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_RETURN = 100.0          # 10,000%
TARGET_MULT = 1.0 + MIN_RETURN  # 101x
MIN_BARS = 60
MIN_ENTRY_PRICE = 1.0       # flag, not auto-drop

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 80)
pd.options.mode.copy_on_write = True

print("DATA_DIR:", DATA_DIR)
print("OUT_DIR:", OUT_DIR)
print(f"Target: {MIN_RETURN:.0f}%  ({TARGET_MULT:.0f}x close)")
assert DATA_DIR.exists(), f"Missing parquet dir: {DATA_DIR}"


In [ ]:
def first_series(frame, name):
    s = frame[name]
    return s.iloc[:, 0] if isinstance(s, pd.DataFrame) else s

def normalize_stock(df, source_name):
    cols = {str(c).lower().strip(): c for c in df.columns}
    aliases = {
        "date": ["date", "datetime", "timestamp"],
        "symbol": ["symbol", "ticker", "security", "stock"],
        "open": ["open"],
        "high": ["high"],
        "low": ["low"],
        "close": ["close", "adj close", "adj_close"],
        "volume": ["volume", "vol", "qty", "quantity", "total traded quantity"],
    }
    out = {}
    for target, names in aliases.items():
        found = next((cols[n] for n in names if n in cols), None)
        if found is not None:
            out[target] = found
    required = ["date", "open", "high", "low", "close"]
    missing = [x for x in required if x not in out]
    if missing:
        raise ValueError(f"{source_name}: missing {missing}; columns={list(df.columns)}")
    symbol = first_series(df, out["symbol"]) if "symbol" in out else Path(source_name).stem
    x = pd.DataFrame({
        "Date": pd.to_datetime(first_series(df, out["date"]), errors="coerce"),
        "Symbol": pd.Series(symbol, index=df.index).astype(str).str.upper().str.strip(),
        "High": pd.to_numeric(first_series(df, out["high"]), errors="coerce"),
        "Low": pd.to_numeric(first_series(df, out["low"]), errors="coerce"),
        "Close": pd.to_numeric(first_series(df, out["close"]), errors="coerce"),
    })
    x = x.dropna(subset=["Date", "Close"])
    x["Date"] = pd.to_datetime(x["Date"], errors="coerce", utc=True).dt.tz_convert(None)
    x = x.loc[x["Close"] > 0]
    return x.sort_values("Date").drop_duplicates(["Symbol", "Date"], keep="last")

def max_dd_on_path(close, high, low):
    c = np.asarray(close, dtype=np.float64)
    h = np.asarray(high, dtype=np.float64)
    lo = np.asarray(low, dtype=np.float64)
    peak_c = np.maximum.accumulate(c)
    dd_c = np.nanmin(c / np.maximum(peak_c, 1e-12) - 1.0)
    peak_hl = np.maximum.accumulate(np.fmax(c, np.where(np.isfinite(h), h, c)))
    trough = np.where(np.isfinite(lo), lo, c)
    dd_l = np.nanmin(trough / np.maximum(peak_hl, 1e-12) - 1.0)
    worst_i = int(np.nanargmin(trough / np.maximum(peak_hl, 1e-12) - 1.0))
    return float(dd_c), float(dd_l), worst_i

def expanding_min_10000pct(dates, close, high, low, symbol):
    c = close.astype(np.float64)
    n = len(c)
    if n < MIN_BARS:
        return None
    rmin = np.inf
    rmin_i = -1
    max_mult = 0.0
    for i in range(n):
        px = c[i]
        if not np.isfinite(px) or px <= 0:
            continue
        if px < rmin:
            rmin = px
            rmin_i = i
        if rmin_i >= 0 and rmin > 0:
            max_mult = max(max_mult, px / rmin)
        if rmin_i >= 0 and px >= rmin * TARGET_MULT:
            sl = slice(rmin_i, i + 1)
            dd_c, dd_l, w_i = max_dd_on_path(c[sl], high[sl], low[sl])
            hit_ret = px / rmin - 1.0
            d0 = pd.Timestamp(dates[rmin_i])
            d1 = pd.Timestamp(dates[i])
            return {
                "Symbol": symbol,
                "EntryDate": d0,
                "HitDate": d1,
                "EntryClose": float(rmin),
                "HitClose": float(px),
                "Return": float(hit_ret),
                "ReturnPct": float(hit_ret * 100.0),
                "Multiple": float(px / rmin),
                "TradingDays": int(i - rmin_i),
                "CalendarDays": int((d1 - d0).days),
                "Years": float((d1 - d0).days / 365.25),
                "MaxDrawdownClose": dd_c,
                "MaxDrawdownLow": dd_l,
                "WorstDDDate": pd.Timestamp(dates[rmin_i + w_i]),
                "BarsInSample": n,
                "SampleStart": pd.Timestamp(dates[0]),
                "SampleEnd": pd.Timestamp(dates[-1]),
                "ExpandingMinMaxMultiple": float(max_mult),
                "LowEntryFlag": bool(rmin < MIN_ENTRY_PRICE),
            }
    return None


In [ ]:
stock_files = sorted(DATA_DIR.rglob("*.parquet"))
print("Parquet files:", len(stock_files))
assert stock_files, f"No parquet under {DATA_DIR}"

rows = []
errors = []
n_ok = 0
for i, f in enumerate(stock_files, 1):
    try:
        raw = pd.read_parquet(f)
        x = normalize_stock(raw, f.name)
        del raw
        if x.empty:
            continue
        for sym, g in x.groupby("Symbol", sort=False):
            rec = expanding_min_10000pct(
                g["Date"].to_numpy(),
                g["Close"].to_numpy(),
                g["High"].to_numpy(),
                g["Low"].to_numpy(),
                str(sym),
            )
            if rec is not None:
                rec["SourceFile"] = f.name
                rows.append(rec)
        n_ok += 1
        del x
        if i % 250 == 0:
            gc.collect()
            print(f"scanned {i}/{len(stock_files)} files, hits {len(rows)}")
    except Exception as e:
        errors.append({"file": f.name, "error": str(e)})

gc.collect()
hits = pd.DataFrame(rows)
print("Files read OK:", n_ok)
print("Read errors:", len(errors))
print("Names with ≥ 10,000% from expanding min:", len(hits))
if errors:
    display(pd.DataFrame(errors).head(20))


## Results

Sorted by calendar days (fastest 10,000% from the expanding-min entry). Max drawdown is the worst peak-to-trough **on the way** to the target.


In [ ]:
if hits.empty:
    print("No stock reached 10,000% from its expanding-min close.")
else:
    hits = hits.sort_values(["CalendarDays", "MaxDrawdownLow"]).reset_index(drop=True)
    hits.to_csv(OUT_DIR / "stocks_10000pct_from_expanding_min.csv", index=False)
    hits.to_parquet(OUT_DIR / "stocks_10000pct_from_expanding_min.parquet", index=False)

    view = hits[[
        "Symbol", "EntryDate", "HitDate", "EntryClose", "HitClose",
        "ReturnPct", "Multiple", "TradingDays", "CalendarDays", "Years",
        "MaxDrawdownClose", "MaxDrawdownLow", "WorstDDDate", "LowEntryFlag",
    ]].copy()
    view["MaxDrawdownClose"] = view["MaxDrawdownClose"].map(lambda v: f"{v:.2%}")
    view["MaxDrawdownLow"] = view["MaxDrawdownLow"].map(lambda v: f"{v:.2%}")
    view["ReturnPct"] = view["ReturnPct"].map(lambda v: f"{v:,.0f}%")
    display(view)

    print("\n— Summary (all qualifying names) —")
    print("Count:", len(hits))
    print("Median calendar days:", float(hits["CalendarDays"].median()))
    print("Median trading days:", float(hits["TradingDays"].median()))
    print("Median years:", float(hits["Years"].median()))
    print("Median max DD (close):", f"{hits['MaxDrawdownClose'].median():.2%}")
    print("Median max DD (low):", f"{hits['MaxDrawdownLow'].median():.2%}")
    print("Worst max DD (low):", f"{hits['MaxDrawdownLow'].min():.2%}",
          "on", hits.loc[hits["MaxDrawdownLow"].idxmin(), "Symbol"])
    print("Fastest (calendar days):", int(hits["CalendarDays"].min()),
          hits.loc[hits["CalendarDays"].idxmin(), "Symbol"])
    print("Slowest (calendar days):", int(hits["CalendarDays"].max()),
          hits.loc[hits["CalendarDays"].idxmax(), "Symbol"])
    print("Low-entry flags (EntryClose <", MIN_ENTRY_PRICE, "):", int(hits["LowEntryFlag"].sum()))


In [ ]:
if not hits.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].hist(hits["CalendarDays"], bins=min(40, max(5, len(hits))), color="steelblue", edgecolor="white")
    axes[0].set_title("Calendar days to +10,000%")
    axes[0].set_xlabel("Days")
    axes[1].hist(hits["MaxDrawdownLow"] * 100, bins=min(40, max(5, len(hits))), color="indianred", edgecolor="white")
    axes[1].set_title("Max DD on the path (using lows)")
    axes[1].set_xlabel("Drawdown %")
    axes[2].scatter(hits["CalendarDays"], hits["MaxDrawdownLow"] * 100, alpha=0.7)
    axes[2].set_xlabel("Calendar days")
    axes[2].set_ylabel("Max DD % (lows)")
    axes[2].set_title("Time vs drawdown endured")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "days_and_drawdown.png", dpi=120)
    plt.show()
    plt.close(fig)

    print("CSV:", OUT_DIR / "stocks_10000pct_from_expanding_min.csv")


## How to read a row

You would have bought at `EntryClose` on `EntryDate` (the lowest close up to that point). You first sat on a +10,000% close on `HitDate`. Between those two dates the position’s worst peak-to-trough was `MaxDrawdownLow` / `MaxDrawdownClose`. That is the drawdown required **to capture this particular 10,000%**, not the stock’s lifetime max drawdown after you were already a hundred-bagger.
